# Procesamiento en streaming con Spark: ventanas, watermarking y semántica de entrega

# Inicializar SparkSession con el conector de Kafka

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("criptoanalytics-s08-streaming")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")
    .config("spark.sql.shuffle.partitions", "3")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# Lectura en modo streaming con Esquema Explícito

Definir el esquema correspondiente al contrato crypto.trade generado por el bridge-binance:

In [2]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StringType, DoubleType, LongType, BooleanType

## Esquema explícito según CONTRATO.md

In [3]:
esquema_crypto = (
    StructType()
    .add("tipoEvento", StringType())
    .add("tradeId", LongType())
    .add("symbol", StringType())
    .add("price", DoubleType())
    .add("quantity", DoubleType())
    .add("quoteQuantity", DoubleType())
    .add("isBuyerMaker", BooleanType())
    .add("origen", StringType())
    .add("timestamp", LongType())
)

## Lectura del topic en modo streaming

In [4]:
stream_crudo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "binance-btc-trades")
    .option("startingOffsets", "latest")
    .load()
)

## Parseo del JSON a columnas tipadas y conversión del timestamp a SQL Timestamp

In [5]:
eventos_crypto = (
    stream_crudo.selectExpr("CAST(value AS STRING) AS json_str")
    .select(from_json(col("json_str"), esquema_crypto).alias("e"))
    .select("e.*")
    .withColumn("ts", (col("timestamp") / 1000).cast("timestamp"))
)

eventos_crypto.printSchema()

root
 |-- tipoEvento: string (nullable = true)
 |-- tradeId: long (nullable = true)
 |-- symbol: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: double (nullable = true)
 |-- quoteQuantity: double (nullable = true)
 |-- isBuyerMaker: boolean (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- ts: timestamp (nullable = true)



# Agregación por Ventana de Tiempo y Watermarking

Métrica de negocio: Precio promedio de BTC (price), Volumen acumulado en USDT (quoteQuantity) y Cantidad de trades (count) por ventanas de 10 segundos con un watermark de 15 segundos.

En mercados de alta frecuencia como Binance BTC/USDT, la latencia de red suele ser inferior a 1-2 segundos. Fijar un watermark de 15 segundos tolera pequeños retrasos sin saturar la memoria acumulando ventanas abiertas indefinidamente.

In [6]:
from pyspark.sql.functions import window, avg, sum, count, col

# Agregación por ventana con watermark de 15 segundos
agregado_crypto = (
    eventos_crypto
    .withWatermark("ts", "15 seconds")
    .groupBy(window(col("ts"), "10 seconds"), col("symbol"))
    .agg(
        avg("price").alias("precio_promedio"),
        sum("quoteQuantity").alias("volumen_usdt"),
        count("*").alias("num_trades")
    )
)

# Iniciar consulta con Sink en memoria
query_agregado = (
    agregado_crypto.writeStream
    .outputMode("update")
    .format("memory")
    .queryName("agg_crypto_memory")
    .option("checkpointLocation", "/tmp/chk_watermark_crypto_v2")
    .trigger(processingTime="5 seconds")
    .start()
)

# Dejar procesar datos en vivo por 25 segundos
query_agregado.awaitTermination(25)

# Evidencia visible directamente en la celda del notebook
print("=== VENTANAS DE AGREGACIÓN PROCESADAS EN MEMORIA ===")
spark.sql("SELECT window.start, window.end, symbol, precio_promedio, volumen_usdt, num_trades FROM agg_crypto_memory ORDER BY window.start DESC").show(10, truncate=False)

query_agregado.stop()

=== VENTANAS DE AGREGACIÓN PROCESADAS EN MEMORIA ===
+-------------------+-------------------+-------+-----------------+------------------+----------+
|start              |end                |symbol |precio_promedio  |volumen_usdt      |num_trades|
+-------------------+-------------------+-------+-----------------+------------------+----------+
|2026-10-09 00:59:30|2026-10-09 00:59:40|BTCUSDT|81806.01         |1999.3388843999999|8         |
|2026-10-09 00:59:20|2026-10-09 00:59:30|BTCUSDT|81804.61771929823|8368.624840399993 |57        |
|2026-10-09 00:59:20|2026-10-09 00:59:30|BTCUSDT|81805.25361904761|13699.104425799993|105       |
|2026-10-09 00:59:20|2026-10-09 00:59:30|BTCUSDT|81805.09747126435|10679.644600399992|87        |
|2026-10-09 00:59:10|2026-10-09 00:59:20|BTCUSDT|81804.3025       |359.1208883       |4         |
|2026-10-09 00:59:10|2026-10-09 00:59:20|BTCUSDT|81804.30692307692|993.1042458       |13        |
+-------------------+-------------------+-------+----------------

# Evento Tardío Descartado por Watermark

Publicar un trade en el pasado (por ejemplo, 45 segundos atrás) cuando la consulta ya haya avanzado con datos en vivo para comprobar que Spark lo descarta.   

In [7]:
#!pip install kafka-python-ng

In [8]:
import builtins
import json
import threading
import time
from kafka import KafkaProducer


def publicar_trade_tardio():
  time.sleep(10)  # Esperar a que el watermark avance con datos en vivo
  producer = KafkaProducer(bootstrap_servers='kafka:9092')

  ts_pasado = int((time.time() - 45) * 1000)  # Evento con 45 segundos de retraso
  trade_tardio = {
      'tipoEvento': 'crypto.trade',
      'tradeId': 9999999999,
      'symbol': (
          'BTCUSDT-TARDIO'  # Símbolo único para distinguir en la consulta SQL
      ),
      'price': 85000.0,
      'quantity': 1.0,
      'quoteQuantity': 85000.0,
      'isBuyerMaker': True,
      'origen': 'prueba-tardio',
      'timestamp': ts_pasado,
  }

  producer.send(
      'binance-btc-trades',
      key=b'BTCUSDT-TARDIO',
      value=json.dumps(trade_tardio).encode('utf-8'),
  )
  producer.flush()
  print(
      '>>> Trade tardío inyectado exitosamente con symbol = BTCUSDT-TARDIO (45s'
      ' en el pasado)'
  )


threading.Thread(target=publicar_trade_tardio, daemon=True).start()

# Definir la consulta streaming dirigida a tabla en memoria
query_tardio = (
    agregado_crypto.writeStream.outputMode('update')
    .format('memory')
    .queryName('agg_tardio')
    .option('checkpointLocation', '/tmp/chk_tardio_crypto_v2')
    .trigger(processingTime='4 seconds')
    .start()
)

query_tardio.awaitTermination(35)

# 1. Métrica cuantitativa de filas descartadas usando builtins.sum
descartadas = builtins.sum(
    p['stateOperators'][0]['numRowsDroppedByWatermark']
    for p in query_tardio.recentProgress
    if p['stateOperators']
)
print(
    f'\n[EVIDENCIA 1] Total de filas descartadas por el watermark: {descartadas}'
)

# 2. Verificación SQL: Debe salir una tabla vacía (0 filas)
print(
    '\n[EVIDENCIA 2] Búsqueda del evento tardío en la tabla agregada (debe salir'
    ' vacío):'
)
spark.sql("SELECT * FROM agg_tardio WHERE symbol = 'BTCUSDT-TARDIO'").show()

query_tardio.stop()

>>> Trade tardío inyectado exitosamente con symbol = BTCUSDT-TARDIO (45s en el pasado)

[EVIDENCIA 1] Total de filas descartadas por el watermark: 1

[EVIDENCIA 2] Búsqueda del evento tardío en la tabla agregada (debe salir vacío):
+------+------+---------------+------------+----------+
|window|symbol|precio_promedio|volumen_usdt|num_trades|
+------+------+---------------+------------+----------+
+------+------+---------------+------------+----------+



# Deduplicación Acotada por Watermark

Filtrar reintentos o mensajes duplicados mediante dropDuplicates acotado por withWatermark en los campos symbol y tradeId.   

In [9]:
# Deduplicación acotada por watermark sobre llaves de evento únicas
deduplicado_crypto = (
    eventos_crypto
    .withWatermark("ts", "30 seconds")
    .dropDuplicatesWithinWatermark(["symbol", "tradeId"])
)

def publicar_duplicados():
    time.sleep(5)
    producer = KafkaProducer(bootstrap_servers='kafka:9092')
    
    trade_dup = {
        "tipoEvento": "crypto.trade",
        "tradeId": 7777777777,
        "symbol": "BTCUSDT",
        "price": 84000.0,
        "quantity": 0.5,
        "quoteQuantity": 42000.0,
        "isBuyerMaker": False,
        "origen": "prueba-dedup",
        "timestamp": int(time.time() * 1000)
    }
    
    # Inyectar el mismo mensaje dos veces en Kafka
    for i in range(2):
        producer.send("binance-btc-trades", key=b"BTCUSDT", value=json.dumps(trade_dup).encode('utf-8'))
    producer.flush()
    print(">>> Trade id=7777777777 enviado 2 veces a Kafka")

threading.Thread(target=publicar_duplicados, daemon=True).start()

# Iniciar consulta streaming a memoria
query_dedup = (
    deduplicado_crypto.writeStream
    .outputMode("append")
    .format("memory")
    .queryName("dedup_memory")
    .option("checkpointLocation", "/tmp/chk_dedup_crypto_v2")
    .trigger(processingTime="3 seconds")
    .start()
)

query_dedup.awaitTermination(25)

# Evidencia: Confirmar que exactamente 1 fila pasó el filtro en lugar de 2
print("\n=== EVIDENCIA DE DEDUPLICACIÓN EN MEMORIA (Debe salir exactamente 1 fila) ===")
spark.sql("SELECT tradeId, symbol, price, quantity, ts FROM dedup_memory WHERE tradeId = 7777777777").show(truncate=False)

query_dedup.stop()

>>> Trade id=7777777777 enviado 2 veces a Kafka

=== EVIDENCIA DE DEDUPLICACIÓN EN MEMORIA (Debe salir exactamente 1 fila) ===
+----------+-------+-------+--------+-----------------------+
|tradeId   |symbol |price  |quantity|ts                     |
+----------+-------+-------+--------+-----------------------+
|7777777777|BTCUSDT|84000.0|0.5     |2026-10-09 01:00:18.485|
+----------+-------+-------+--------+-----------------------+



# Comparación de Intervalos de Disparo / Trigger

Comparar el comportamiento en cuanto a micro-lotes procesados en el mismo lapso de tiempo usando un trigger rápido (1 segundo) frente a uno más lento (5 segundos).   

In [10]:
def medir_batch_id(intervalo_str, segundos_ejecucion):
    query = (
        eventos_crypto.writeStream
        .outputMode("append")
        .format("memory")
        .queryName(f"tabla_{intervalo_str.replace(' ', '_')}")
        .trigger(processingTime=intervalo_str)
        .start()
    )
    time.sleep(segundos_ejecucion)
    query.stop()
    return query.lastProgress["batchId"] if query.lastProgress else None

batch_1s = medir_batch_id("1 second", 15)
batch_5s = medir_batch_id("5 seconds", 15)

print(f"BatchID alcanzado con Trigger de 1s en 15 segundos: {batch_1s}")
print(f"BatchID alcanzado con Trigger de 5s en 15 segundos: {batch_5s}")

BatchID alcanzado con Trigger de 1s en 15 segundos: 15
BatchID alcanzado con Trigger de 5s en 15 segundos: 3


# Persistencia en Parquet con Checkpointing

Persistir el stream en archivos Parquet en modo append y verificar que múltiples ejecuciones acumulen datos en lugar de sobreescribirlos.   

In [13]:
PATH_PARQUET = "/home/jovyan/work/artifacts/btc_trades_parquet"
PATH_CHECKPOINT = "/home/jovyan/work/artifacts/chk_btc_parquet"

query_parquet = (
    eventos_crypto.writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", PATH_PARQUET)
    .option("checkpointLocation", PATH_CHECKPOINT)
    .trigger(processingTime="5 seconds")
    .start()
)

query_parquet.awaitTermination(20)
query_parquet.stop()

## Primera verificación

In [14]:
df_guardado_1 = spark.read.parquet(PATH_PARQUET)
conteo_1 = df_guardado_1.count()
print(f"Conteo tras corrida 1: {conteo_1} filas guardadas.")

Conteo tras corrida 1: 411 filas guardadas.


## Segunda corrida para verificar acumulación

In [15]:
query_parquet_2 = (
    eventos_crypto.writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", PATH_PARQUET)
    .option("checkpointLocation", PATH_CHECKPOINT)
    .trigger(processingTime="5 seconds")
    .start()
)
query_parquet_2.awaitTermination(20)
query_parquet_2.stop()

## Segunda verificación

In [16]:
df_guardado_2 = spark.read.parquet(PATH_PARQUET)
conteo_2 = df_guardado_2.count()
print(f"Conteo tras corrida 2: {conteo_2} filas guardadas (Acumulado correctamente).")

Conteo tras corrida 2: 770 filas guardadas (Acumulado correctamente).
